# Hybrid Neural Collaborative Filtering
## Imports

In [ ]:
import torch
import pandas as pd
from pathlib import Path
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import random
import optuna # for Grid Search

## Loading csv files

In [ ]:
### Interactions
interactions = pd.read_csv(Path.cwd().parent/"data"/"interactions_train.csv")
n_users = interactions["u"].nunique()
n_items = interactions["i"].nunique()
interactions = interactions.sort_values(["u", "t"])
interactions["pct_rank"] = interactions.groupby("u")["t"].rank(pct=True, method='dense')
interactions.reset_index(inplace=True, drop=True)
train_data = interactions[interactions["pct_rank"] < 0.8]
test_data = interactions[interactions["pct_rank"] >= 0.8]

# Books
items = pd.read_csv(Path.cwd().parent/"data"/"items.csv")

## Dataset with Negative Sampling

In [ ]:
class LibraryDataset(Dataset):
    def __init__(self, user_tensor, item_tensor, labels_tensor):
        """
        Expects PyTorch tensors for users, items, and labels (1 for read, 0 for not read).
        """
        self.users = user_tensor
        self.items = item_tensor
        self.labels = labels_tensor

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.labels[idx]

class BookInteractionDataset(Dataset):
    def __init__(self, df, num_items, num_negatives=4):
        """
        df: Pandas dataframe with 'u' (user) and 'i' (item) columns
        num_items: Total number of unique items in the dataset
        num_negatives: How many negative samples to generate per positive interaction
        """
        self.users = df['u'].values
        self.items = df['i'].values
        self.num_items = num_items
        self.num_negatives = num_negatives
        
        # Group positive items by user for fast lookup to avoid sampling true positives as negatives
        self.user_positives = df.groupby('u')['i'].apply(set).to_dict()

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        user = self.users[idx]
        pos_item = self.items[idx]
        
        # We will return the user, the positive item, and a list of negative items
        neg_items = []
        for _ in range(self.num_negatives):
            neg_item = random.randint(0, self.num_items - 1)
            while neg_item in self.user_positives[user]:
                neg_item = random.randint(0, self.num_items - 1)
            neg_items.append(neg_item)
            
        # Convert to tensors
        # Return format: user, positive item, array of negative items
        return torch.tensor(user, dtype=torch.long), \
               torch.tensor(pos_item, dtype=torch.long), \
               torch.tensor(neg_items, dtype=torch.long)

## Hybrid Neural Recommender Model

In [ ]:
class HybridBookRecommender(nn.Module):
    def __init__(self, num_users, num_items, embed_dim, text_embed_dim):
        super(HybridBookRecommender, self).__init__()
        
        # 1. Collaborative Filtering Embeddings (Learnable)
        self.user_embedding = nn.Embedding(num_embeddings=num_users, embedding_dim=embed_dim)
        self.item_embedding = nn.Embedding(num_embeddings=num_items, embedding_dim=embed_dim)
        
        # 2. Multi-Layer Perceptron (MLP) for combining signals
        # Input size = User Embed + Item Embed + Hugging Face Text Embed
        mlp_input_dim = embed_dim + embed_dim + text_embed_dim
        
        self.mlp = nn.Sequential(
            nn.Linear(mlp_input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        
        # 3. Final Prediction Layer
        self.output_layer = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, user_idx, item_idx, item_text_embeddings):
        # Get CF embeddings
        u_embed = self.user_embedding(user_idx)
        i_embed = self.item_embedding(item_idx)
        
        # Concatenate CF embeddings with the Hugging Face text embeddings
        # item_text_embeddings should be pre-computed and passed in the batch
        concat_features = torch.cat([u_embed, i_embed, item_text_embeddings], dim=-1)
        
        # Pass through MLP
        mlp_out = self.mlp(concat_features)
        
        # Predict probability of interaction (0 to 1)
        prediction = self.sigmoid(self.output_layer(mlp_out))
        return prediction.squeeze()

class HybridBookRecommender(nn.Module): # for Optuna
    def __init__(self, num_users, num_items, embed_dim, text_embed_dim, dropout_rate=0.2):
        super(HybridBookRecommender, self).__init__()
        
        self.user_embedding = nn.Embedding(num_embeddings=num_users, embedding_dim=embed_dim)
        self.item_embedding = nn.Embedding(num_embeddings=num_items, embedding_dim=embed_dim)
        
        mlp_input_dim = embed_dim + embed_dim + text_embed_dim
        
        self.mlp = nn.Sequential(
            nn.Linear(mlp_input_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate), # Tunable dropout
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate), # Tunable dropout
            nn.Linear(64, 32),
            nn.ReLU()
        )
        
        self.output_layer = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, user_idx, item_idx, item_text_embeddings):
        u_embed = self.user_embedding(user_idx)
        i_embed = self.item_embedding(item_idx)
        concat_features = torch.cat([u_embed, i_embed, item_text_embeddings], dim=-1)
        mlp_out = self.mlp(concat_features)
        return self.sigmoid(self.output_layer(mlp_out)).squeeze()

class HybridBookRecommender(nn.Module):
    def __init__(self, num_users, num_items, embed_dim, text_embed_dim, num_numerical_features=2, dropout_rate=0.2):
        super(HybridBookRecommender, self).__init__()
        
        self.user_embedding = nn.Embedding(num_embeddings=num_users, embedding_dim=embed_dim)
        self.item_embedding = nn.Embedding(num_embeddings=num_items, embedding_dim=embed_dim)
        
        # INCREASED DIMENSION: CF Embeds + Text Embeds + Numerical Features
        mlp_input_dim = embed_dim + embed_dim + text_embed_dim + num_numerical_features
        
        self.mlp = nn.Sequential(
            nn.Linear(mlp_input_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        
        self.output_layer = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, user_idx, item_idx, item_text_embeddings, item_numerical_features):
        u_embed = self.user_embedding(user_idx)
        i_embed = self.item_embedding(item_idx)
        
        # Concatenate everything into one massive feature vector
        concat_features = torch.cat([u_embed, i_embed, item_text_embeddings, item_numerical_features], dim=-1)
        
        mlp_out = self.mlp(concat_features)
        return self.sigmoid(self.output_layer(mlp_out)).squeeze()

## Training Loop
### Initialization

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# Assuming n_users and n_items are defined, and you have pre-computed text embeddings
# text_embeddings_tensor = torch.load("item_text_embeddings.pt").to(device)

embed_dim = 64
text_embed_dim = 384 # 384 for paraphrase-multilingual-MiniLM-L12-v2, 786 for multilingual-e5-base and EmbeddingGemma, 1024 for bge-m3

# Initialize your model (from previous step) and move it to the GPU
model = HybridBookRecommender(n_users, n_items, embed_dim, text_embed_dim).to(device)

# Binary Cross Entropy Loss and AdamW optimizer
criterion = nn.BCELoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

# Create DataLoaders
# train_data is the 80% split dataframe, val_data is the 20% remaining
train_dataset = BookInteractionDataset(train_data, num_items=n_items, num_negatives=4)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=2)
val_dataset = BookInteractionDataset(df=test_data, num_items=n_items, num_negatives=4)
val_loader = DataLoader(dataset=val_dataset, batch_size=256, shuffle=False, num_workers=2)


### Training Loop

In [ ]:
def objective(trial):
    # --- 1. Define the Hyperparameter Search Space ---
    # We ask Optuna to suggest values for these parameters
    embed_dim = trial.suggest_categorical('embed_dim', [32, 64, 128])
    lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    
    # --- 2. Initialize Model and Optimizer with Suggested Params ---
    model = HybridBookRecommender(
        num_users=n_users, 
        num_items=n_items, 
        embed_dim=embed_dim, 
        text_embed_dim=text_embed_dim, 
        dropout_rate=dropout_rate
    ).to(device)
    
    criterion = nn.BCELoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    # Keep the epochs low (e.g., 3-5) during the Optuna search to save time. 
    # We will train the best model for longer later.
    epochs_for_tuning = 3 
    
    # --- 3. Training Loop ---
    for epoch in range(epochs_for_tuning):
        model.train()
        for users, pos_items, neg_items in train_loader: # (Assuming train_loader is defined globally)
            users = users.to(device)
            pos_items = pos_items.to(device)
            neg_items = neg_items.to(device)
            
            batch_users = users.repeat_interleave(1 + train_dataset.num_negatives)
            batch_items = torch.cat([pos_items.unsqueeze(1), neg_items], dim=1).flatten()
            
            pos_labels = torch.ones_like(pos_items, dtype=torch.float32)
            neg_labels = torch.zeros_like(neg_items, dtype=torch.float32)
            batch_labels = torch.cat([pos_labels.unsqueeze(1), neg_labels], dim=1).flatten().to(device)
            
            # Dummy text features (replace with real pre-computed features later)
            dummy_text_features = torch.zeros((len(batch_items), text_embed_dim)).to(device)
            
            optimizer.zero_grad()
            predictions = model(batch_users, batch_items, dummy_text_features)
            loss = criterion(predictions, batch_labels)
            loss.backward()
            optimizer.step()
            
    # --- 4. Validation Loop ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for users, pos_items, neg_items in val_loader: # (Assuming val_loader is defined)
            users = users.to(device)
            pos_items = pos_items.to(device)
            neg_items = neg_items.to(device)
            
            batch_users = users.repeat_interleave(1 + val_dataset.num_negatives)
            batch_items = torch.cat([pos_items.unsqueeze(1), neg_items], dim=1).flatten()
            
            pos_labels = torch.ones_like(pos_items, dtype=torch.float32)
            neg_labels = torch.zeros_like(neg_items, dtype=torch.float32)
            batch_labels = torch.cat([pos_labels.unsqueeze(1), neg_labels], dim=1).flatten().to(device)
            
            dummy_text_features = torch.zeros((len(batch_items), text_embed_dim)).to(device)
            
            predictions = model(batch_users, batch_items, dummy_text_features)
            loss = criterion(predictions, batch_labels)
            val_loss += loss.item()
            
    avg_val_loss = val_loss / len(val_loader)
    
    # Optuna needs to know how we did so it can guess better next time!
    return avg_val_loss

# Create a study object and specify the direction is 'minimize' (since we are returning BCE Loss)
study = optuna.create_study(direction='minimize', study_name="Hybrid_Recommender_Tuning")

# Run 20 trials. You can increase this if you have time!
study.optimize(objective, n_trials=20)

print("\n--- Tuning Complete ---")
print(f"Best Trial Score (Validation Loss): {study.best_trial.value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_trial.params.items():
    print(f"  {key}: {value}")

In [ ]:
# num_epochs = 10

# for epoch in range(num_epochs):
#     model.train()
#     total_loss = 0.0
    
#     # Progress bar for the epoch
#     loop = tqdm(train_loader, leave=False, desc=f"Epoch {epoch+1}/{num_epochs}")
    
#     for users, pos_items, neg_items in loop:
#         # Move inputs to GPU
#         users = users.to(device)
#         pos_items = pos_items.to(device)
#         neg_items = neg_items.to(device)
        
#         # --- Prepare the batch data ---
#         # We need to combine the positive (label 1) and negative (label 0) interactions
#         # users shape: [batch_size] -> repeat it to match negatives: [batch_size * (1 + num_negatives)]
#         batch_users = users.repeat_interleave(1 + train_dataset.num_negatives)
        
#         # items shape: concatenate positives and flattened negatives
#         batch_items = torch.cat([pos_items.unsqueeze(1), neg_items], dim=1).flatten()
        
#         # labels: 1s for positives, 0s for negatives
#         pos_labels = torch.ones_like(pos_items, dtype=torch.float32)
#         neg_labels = torch.zeros_like(neg_items, dtype=torch.float32)
#         batch_labels = torch.cat([pos_labels.unsqueeze(1), neg_labels], dim=1).flatten().to(device)
        
#         # --- Fetch the corresponding text embeddings ---
#         # (Assuming you created a global tensor `text_embeddings_tensor` of shape [num_items, text_embed_dim])
#         # batch_text_features = text_embeddings_tensor[batch_items]
        
#         # --- Forward Pass ---
#         optimizer.zero_grad()
        
#         # Replace `batch_text_features` below with the actual fetched text tensor
#         # For testing without text features yet, you could pass torch.zeros for now
#         dummy_text_features = torch.zeros((len(batch_items), text_embed_dim)).to(device) 
        
#         predictions = model(batch_users, batch_items, dummy_text_features)
        
#         # --- Backward Pass ---
#         loss = criterion(predictions, batch_labels)
#         loss.backward()
#         optimizer.step()
        
#         total_loss += loss.item()
#         loop.set_postfix(loss=loss.item())
        
#     avg_loss = total_loss / len(train_loader)
#     print(f"Epoch {epoch+1}/{num_epochs} - Average Loss: {avg_loss:.4f}")

#     # (Optional but recommended)
#     # Put a validation function here to calculate Precision@10 on your 20% test_data

## Final Training on the Full Dataset 

## Create the Full Dataset & DataLoader + Initialize Final Model

In [ ]:
best_params = study.best_trial.params
print("Training with best parameters:", best_params)

full_dataset = BookInteractionDataset(interactions, num_items=n_items, num_negatives=4)
full_loader = DataLoader(full_dataset, batch_size=256, shuffle=True, num_workers=2)

final_model = HybridBookRecommender(num_users=n_users, num_items=n_items, embed_dim=best_params['embed_dim'], text_embed_dim=text_embed_dim, dropout_rate=best_params['dropout_rate']).to(device)

criterion = nn.BCELoss()
optimizer = optim.AdamW(final_model.parameters(), lr=best_params['lr'], weight_decay=best_params['weight_decay'])

### Final Training Loop

In [ ]:
num_epochs = 15
final_model.train()

for epoch in range(num_epochs):
    total_loss = 0.0
    loop = tqdm(full_loader, leave=False, desc=f"Final Training Epoch {epoch+1}/{num_epochs}")
    
    for users, pos_items, neg_items in loop:
        users = users.to(device)
        pos_items = pos_items.to(device)
        neg_items = neg_items.to(device)
        
        batch_users = users.repeat_interleave(1 + full_dataset.num_negatives)
        batch_items = torch.cat([pos_items.unsqueeze(1), neg_items], dim=1).flatten()
        
        pos_labels = torch.ones_like(pos_items, dtype=torch.float32)
        neg_labels = torch.zeros_like(neg_items, dtype=torch.float32)
        batch_labels = torch.cat([pos_labels.unsqueeze(1), neg_labels], dim=1).flatten().to(device)
        
        # IMPORTANT: Replace dummy_text_features with your actual Hugging Face embeddings
        # batch_text_features = text_embeddings_tensor[batch_items]
        dummy_text_features = torch.zeros((len(batch_items), text_embed_dim)).to(device)
        
        optimizer.zero_grad()
        predictions = final_model(batch_users, batch_items, dummy_text_features)
        
        loss = criterion(predictions, batch_labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())
        
    print(f"Epoch {epoch+1}/{num_epochs} Complete - Loss: {total_loss / len(full_loader):.4f}")

### Generating the Submission


In [ ]:
# 1. Load the sample submission format
submission_path = Path.cwd().parent / "data" / "sample_submission.csv"
sample_submission = pd.read_csv(submission_path)

# 2. Precompute user histories for fast filtering
user_histories = interactions.groupby('u')['i'].apply(set).to_dict()

# If using real text embeddings, load them to device once
# all_text_features = text_embeddings_tensor.to(device)

predictions_list = []
final_model.eval()

# 3. Inference Loop
print("Generating top 10 recommendations for each user...")
with torch.no_grad():
    for user in tqdm(sample_submission["user_id"]):
        
        # Get items the user has already interacted with
        read_items = user_histories.get(user, set())
        
        # Find all unread items
        unread_items = [i for i in range(n_items) if i not in read_items]
        unread_items_tensor = torch.tensor(unread_items, dtype=torch.long).to(device)
        
        # Create a user tensor of the same size
        user_tensor = torch.tensor([user] * len(unread_items), dtype=torch.long).to(device)
        
        # Get text features for unread items
        # unread_text_features = all_text_features[unread_items_tensor]
        dummy_features = torch.zeros((len(unread_items), text_embed_dim)).to(device)
        
        # Predict scores for all unread items simultaneously
        scores = final_model(user_tensor, unread_items_tensor, dummy_features)
        
        # Get the indices of the top 10 scores
        # Since 'scores' corresponds to the filtered 'unread_items_tensor', 
        # we need to map the indices back to the actual Item IDs
        top_10_local_idx = torch.topk(scores, 10).indices
        top_10_item_ids = unread_items_tensor[top_10_local_idx].cpu().numpy()
        
        # Format as space-separated string
        predictions_list.append(" ".join(map(str, top_10_item_ids)))

# 4. Save Submission
sample_submission["recommendation"] = predictions_list

output_path = Path.cwd().parent / "submissions" / "submission_hybrid_ncf.csv"
output_path.parent.mkdir(parents=True, exist_ok=True) # Ensure directory exists
sample_submission.to_csv(output_path, index=False)

print(f"Submission saved successfully to {output_path}!")